# 🔍 Notebook 2: Finding the Difference Between Two Replicas

Now suppose two replicas of a database have *almost* the same data — say, 1 million keys each, with maybe a few hundred differing. We want to figure out which keys differ, with as little traffic as possible.

We compare three approaches:

1. 🟥 **BAD** — send all keys/values from one replica to the other. O(N) traffic.
2. 🟨 **OK** — send a hash of every key. Still O(N) traffic but smaller per key.
3. 🟩 **BEST** — Merkle tree comparison. O(K log N) traffic where K is the number of differences.

## Learning objectives
- Compare these three strategies on the same dataset.
- See why Merkle wins when the differences are few.

In [ ]:
import hashlib, random
random.seed(0)

def H(b): return hashlib.sha256(b).digest()

# Build two replicas: 1024 keys each, with a small number of differences.
N = 1024
DIFFS = 5                                    # how many keys actually differ

base = [(f"key{i}", f"value{i}") for i in range(N)]
replica_A = list(base)
replica_B = list(base)
diff_indices = random.sample(range(N), DIFFS)
for i in diff_indices:
    k, v = replica_B[i]
    replica_B[i] = (k, v + "_modified")

print(f"{N} keys, {DIFFS} differ at indices {sorted(diff_indices)}")

In [ ]:
# 🟥 BAD: send all key/value pairs across the wire
def bad_diff(A, B):
    bytes_sent = sum(len(k) + len(v) for k, v in A)        # full payload
    diffs = [k for (k, va), (_, vb) in zip(A, B) if va != vb]
    return diffs, bytes_sent

diffs, bytes_sent = bad_diff(replica_A, replica_B)
print(f"BAD  : found {len(diffs)} diffs, sent {bytes_sent:,} bytes")

In [ ]:
# 🟨 OK: send hash per key
def ok_diff(A, B):
    hashes_A = [H(f"{k}={v}".encode()) for k, v in A]
    hashes_B = [H(f"{k}={v}".encode()) for k, v in B]
    bytes_sent = len(hashes_A) * 32                          # 32 bytes per SHA-256
    diffs = [A[i][0] for i in range(len(A)) if hashes_A[i] != hashes_B[i]]
    return diffs, bytes_sent

diffs, bytes_sent = ok_diff(replica_A, replica_B)
print(f"OK   : found {len(diffs)} diffs, sent {bytes_sent:,} bytes")

In [ ]:
# 🟩 BEST: walk a Merkle tree, only descend into branches that differ.
def merkle(leaves):
    level = [H(x) for x in leaves]
    levels = [level]
    while len(level) > 1:
        if len(level) % 2: level = level + [level[-1]]
        level = [H(level[i] + level[i+1]) for i in range(0, len(level), 2)]
        levels.append(level)
    return levels

def differing_leaf_indices(tree_a, tree_b):
    '''Walk both trees from the root; return indices of leaves that differ.'''
    bytes_sent = [0]

    def walk(level, idx):
        bytes_sent[0] += 32                  # we sent this node's hash over the network
        a = tree_a[level][idx] if idx < len(tree_a[level]) else None
        b = tree_b[level][idx] if idx < len(tree_b[level]) else None
        if a == b: return []
        if level == 0: return [idx]
        return walk(level - 1, 2*idx) + walk(level - 1, 2*idx + 1)

    top = len(tree_a) - 1
    diffs = walk(top, 0)
    return diffs, bytes_sent[0]

leaves_A = [f"{k}={v}".encode() for k, v in replica_A]
leaves_B = [f"{k}={v}".encode() for k, v in replica_B]
tree_A = merkle(leaves_A)
tree_B = merkle(leaves_B)

diff_idx, bytes_sent = differing_leaf_indices(tree_A, tree_B)
print(f"BEST : found {len(diff_idx)} diffs at {sorted(diff_idx)}, sent {bytes_sent:,} bytes")

## 📊 Why Merkle wins for "mostly equal" datasets

If the two replicas are identical → Merkle compares **one** hash. Total traffic ≈ 32 bytes.
If they differ by `K` leaves → Merkle visits `O(K log N)` nodes. The full-replica diff always sends `O(N)` bytes.

Real systems (Cassandra, DynamoDB, Git, IPFS, blockchains, BitTorrent) all use Merkle trees for this reason: cheap to *prove equality*, cheap to *locate differences* when there are few.